# 第 10 章：資料品質檢查
**學習目標**：
- 檢查主鍵重複與資料表之間的參照完整性。
- 驗證數值範圍、缺失值及類別欄位。
- 將業務規則轉換為可執行的品質檢查。
- 辨識選樣偏誤與時間洩漏風險。
- 建立一份容易閱讀的資料品質摘要。

## 1. 環境設定與資料載入

`common.py` 提供課程共用的套件檢查與資料載入函式。先載入本章會使用的五張資料表。

In [1]:
import pandas as pd
from IPython.display import display

#`common.py` 提供課程共用的套件
from common import ensure_packages, load_data

ensure_packages()
data = load_data()

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]
sessions = data["sessions"]
events = data["events"]


In [2]:

# 先確認每張資料表的筆數與欄位數，掌握檢查範圍。
table_shapes = pd.DataFrame(
    {
        "資料表": ["customers", "orders", "order_items", "sessions", "events"],
        "筆數": [len(customers), len(orders), len(order_items), len(sessions), len(events)],
        "欄位數": [customers.shape[1], orders.shape[1], order_items.shape[1], sessions.shape[1], events.shape[1]],
    }
)
display(table_shapes)

,資料表,筆數,欄位數
0,customers,2500,5
1,orders,22000,5
2,order_items,41549,5
3,sessions,70000,7
4,events,232365,6


## 2. 主鍵唯一性檢查

主鍵應能唯一識別每一筆資料。若主鍵重複，合併資料時可能產生非預期的資料膨脹。

In [3]:
# duplicated() 預設將第一次出現保留，之後重複出現者標記為 True。
duplicate_counts = pd.Series(
    {
        "customers.customer_id": int(customers["customer_id"].duplicated().sum()),
        "orders.order_id": int(orders["order_id"].duplicated().sum()),
        "sessions.session_id": int(sessions["session_id"].duplicated().sum()),
    },
    name="重複筆數",
)
display(duplicate_counts.to_frame())

,重複筆數
customers.customer_id,0
orders.order_id,0
sessions.session_id,0


**解讀方式**：理想結果皆為 0。若結果大於 0，應進一步顯示重複列，確認是資料重複匯入、鍵值產生錯誤，或主鍵定義不完整。

## 3. 參照完整性：檢查孤兒資料

「孤兒資料」是指子表中的外鍵，在父表找不到對應主鍵。例如訂單品項找不到訂單，或工作階段找不到客戶。

In [4]:
# 找出無法對應至 orders 的訂單品項。
orphan_items = order_items.loc[
    ~order_items["order_id"].isin(orders["order_id"])
].copy()

# 找出無法對應至 customers 的工作階段。
orphan_sessions = sessions.loc[
    ~sessions["customer_id"].isin(customers["customer_id"])
].copy()

orphan_summary = pd.DataFrame(
    {
        "檢查項目": ["order_items → orders", "sessions → customers"],
        "孤兒筆數": [len(orphan_items), len(orphan_sessions)],
        "孤兒比例": [
            len(orphan_items) / len(order_items) if len(order_items) else 0,
            len(orphan_sessions) / len(sessions) if len(sessions) else 0,
        ],
    }
)
# 將比例轉成百分比文字，不需要安裝 jinja2。
ratio_column = orphan_summary.columns[-1]
orphan_summary[ratio_column] = orphan_summary[ratio_column].map(
    lambda value: f"{value:.4%}"
)
display(orphan_summary)


,檢查項目,孤兒筆數,孤兒比例
0,order_items → orders,0,0.0000%
1,sessions → customers,0,0.0000%


如果孤兒筆數不為 0，可使用 `display(orphan_sessions.head())` 查看問題資料，但正式報表不宜一次輸出全部異常列。

## 4. 數值範圍檢查

依資料定義，單價不得為負數、折扣率應介於 0 到 1、商品數量必須大於 0。與其只使用 `assert` 在第一個錯誤時中止，這裡同時計算每條規則的異常筆數。

In [5]:
range_checks = pd.DataFrame(
    [
        {
            "規則": "unit_price >= 0",
            "異常筆數": int((order_items["unit_price"] < 0).sum()),
        },
        {
            "規則": "0 <= discount_rate <= 1",
            "異常筆數": int((~order_items["discount_rate"].between(0, 1)).sum()),
        },
        {
            "規則": "quantity > 0",
            "異常筆數": int((order_items["quantity"] <= 0).sum()),
        },
    ]
)
range_checks["結果"] = range_checks["異常筆數"].eq(0).map({True: "通過", False: "失敗"})
display(range_checks)

,規則,異常筆數,結果
0,unit_price >= 0,0,通過
1,0 <= discount_rate <= 1,0,通過
2,quantity > 0,0,通過


> 提醒：若欄位含有缺失值，單純比較大小可能不會把缺失值算成違規。因此缺失值應在下一節另行檢查。

## 5. 缺失值報表

將重複使用的邏輯包裝成函式，為每張資料表計算各欄位的缺失筆數與比例，只保留確實有缺失值的欄位。

In [6]:
def null_report(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """回傳指定資料表中，缺失筆數大於 0 的欄位摘要。"""
    null_counts = df.isna().sum()#每個欄位統計空值的數量
    denominator = len(df)
    null_pct = (null_counts / denominator * 100).round(2) if denominator else null_counts.astype(float)

    report = pd.DataFrame(
        {
            "資料表": table_name,
            "欄位": null_counts.index,
            "缺失筆數": null_counts.values,
            "缺失比例(%)": null_pct.values,
        }
    )
    return report.loc[report["缺失筆數"] > 0]


tables = {
    "sessions": sessions,
    "events": events,
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
}

full_null_report = pd.concat( #合併
    [null_report(df, name) for name, df in tables.items()],
    ignore_index=True,
)

if full_null_report.empty:
    print("所有受檢欄位皆無缺失值。")
else:
    display(full_null_report.sort_values("缺失比例(%)", ascending=False))

,資料表,欄位,缺失筆數,缺失比例(%)
0,events,order_id,223675,96.26


缺失值不一定代表錯誤。例如未購買的瀏覽事件，其 `revenue` 可能合理地留空。判斷品質時必須結合欄位定義與業務情境。

## 6. 類別值與業務規則

類別欄位應限制在允許值集合內。本例的合法事件類型是 `page_view`、`add_to_cart` 與 `purchase`。

In [7]:
valid_event_types = {"page_view", "add_to_cart", "purchase"}
actual_event_types = set(events["event_type"].dropna().unique())
invalid_event_types = actual_event_types - valid_event_types

print("資料中的事件類型：", sorted(actual_event_types))
print("不合法的事件類型：", sorted(invalid_event_types))
print("檢查結果：", "通過" if not invalid_event_types else "失敗")

資料中的事件類型： ['add_to_cart', 'page_view', 'purchase']
不合法的事件類型： []
檢查結果： 通過


### 6.1 購買事件的營收規則

依練習題規則，`purchase` 事件的營收不應為空值或 0。

In [8]:
purchase_events = events.loc[events["event_type"] == "purchase"].copy()
invalid_purchase_revenue = purchase_events.loc[
    purchase_events["revenue"].isna() | (purchase_events["revenue"] == 0)
]

print(f"購買事件共 {len(purchase_events):,} 筆。")
print(f"營收為空值或 0 的購買事件：{len(invalid_purchase_revenue):,} 筆。")

購買事件共 8,690 筆。
營收為空值或 0 的購買事件：0 筆。


### 6.2 已取消訂單不應產生正營收

將訂單與正營收事件依 `order_id` 合併，再篩選狀態為 `cancelled` 的訂單。

In [9]:
cancelled_with_revenue = orders.merge(
    events.loc[events["revenue"] > 0],
    on="order_id",
    how="inner",
)
cancelled_with_revenue = cancelled_with_revenue.loc[
    cancelled_with_revenue["status"] == "cancelled"
]

print(f"已取消但仍有正營收的訂單事件：{len(cancelled_with_revenue):,} 筆。")

已取消但仍有正營收的訂單事件：0 筆。


## 7. 分析風險檢查

資料格式正確不代表分析一定可靠。本節進一步檢查裝置組成與轉換率，並確認事件時間順序。

### 7.1 選樣偏誤：裝置組成與轉換率

如果樣本中的裝置比例和真實母體差異很大，整體轉換率可能受到選樣偏誤影響。因此應同時查看流量組成及各裝置轉換率。

In [10]:
purchase_session_ids = set(
    events.loc[events["event_type"] == "purchase", "session_id"]
)

session_quality = sessions.copy()
session_quality["converted"] = session_quality["session_id"].isin(purchase_session_ids).astype(int)

device_summary = pd.DataFrame(
    {
        "工作階段占比(%)": (session_quality["device"].value_counts(normalize=True) * 100).round(2),
        "轉換率(%)": (session_quality.groupby("device")["converted"].mean() * 100).round(2),
    }
).sort_values("轉換率(%)", ascending=False)

display(device_summary)

,工作階段占比(%),轉換率(%)
device,,
desktop,32.93,16.57
mobile,62.06,10.63
tablet,5.01,7.25


### 7.2 時間洩漏：購買不得早於工作階段開始

若購買事件時間早於 `session_start`，可能是時區、解析或資料串接錯誤，也可能在模型訓練中造成時間洩漏。

In [11]:
session_starts = sessions[["session_id", "session_start"]].copy()
session_starts["session_start"] = pd.to_datetime(session_starts["session_start"])

purchase_times = events.loc[
    events["event_type"] == "purchase",
    ["session_id", "event_time"],
].copy()
purchase_times["event_time"] = pd.to_datetime(purchase_times["event_time"])

purchase_timeline = purchase_times.merge(session_starts, on="session_id", how="left")
time_leakage_rows = purchase_timeline.loc[
    purchase_timeline["event_time"] < purchase_timeline["session_start"]
]

print(f"購買早於工作階段開始的資料：{len(time_leakage_rows):,} 筆。")

購買早於工作階段開始的資料：0 筆。


## 8. 彙整資料品質結果

將前面各項檢查整合為一張摘要表，方便快速判斷哪些規則需要追蹤。

In [12]:
quality_summary = pd.DataFrame(
    [
        ("主鍵唯一性", "customer_id 重複", duplicate_counts["customers.customer_id"]),
        ("主鍵唯一性", "order_id 重複", duplicate_counts["orders.order_id"]),
        ("主鍵唯一性", "session_id 重複", duplicate_counts["sessions.session_id"]),
        ("參照完整性", "孤兒訂單品項", len(orphan_items)),
        ("參照完整性", "孤兒工作階段", len(orphan_sessions)),
        ("數值範圍", "違反範圍規則", int(range_checks["異常筆數"].sum())),
        ("類別合法性", "不合法事件類型", len(invalid_event_types)),
        ("業務規則", "購買營收為空或 0", len(invalid_purchase_revenue)),
        ("業務規則", "取消訂單有正營收", len(cancelled_with_revenue)),
        ("時間一致性", "購買早於工作階段", len(time_leakage_rows)),
    ],
    columns=["檢查面向", "檢查項目", "異常筆數"],
)
quality_summary["結果"] = quality_summary["異常筆數"].eq(0).map({True: "通過", False: "需檢查"})
display(quality_summary)

,檢查面向,檢查項目,異常筆數,結果
0,主鍵唯一性,customer_id 重複,0,通過
1,主鍵唯一性,order_id 重複,0,通過
2,主鍵唯一性,session_id 重複,0,通過
3,參照完整性,孤兒訂單品項,0,通過
4,參照完整性,孤兒工作階段,0,通過
5,數值範圍,違反範圍規則,0,通過
6,類別合法性,不合法事件類型,0,通過
7,業務規則,購買營收為空或 0,0,通過
8,業務規則,取消訂單有正營收,0,通過
9,時間一致性,購買早於工作階段,0,通過


## 9. 練習題

請新增一條規則：每筆事件的 `session_id` 都必須存在於 `sessions` 資料表。

完成後請回答：孤兒事件有幾筆？占全部事件的比例是多少？

In [13]:
# 練習提示：使用 isin() 建立布林條件。
# TODO：可先遮住以下參考答案，再自行完成。
orphan_events = events.loc[
    ~events["session_id"].isin(sessions["session_id"])
]
orphan_event_ratio = len(orphan_events) / len(events) if len(events) else 0

print(f"孤兒事件：{len(orphan_events):,} 筆")
print(f"孤兒事件比例：{orphan_event_ratio:.4%}")

孤兒事件：0 筆
孤兒事件比例：0.0000%


## 常見錯誤與延伸

**常見錯誤**：只計算異常筆數，卻沒有保留異常資料。實務上應同時產生摘要與少量問題樣本，才能追查原因。

**另一個陷阱**：直接使用 `assert` 作為正式品質報表。`assert` 適合在不符合條件時立即中止，但不利於一次蒐集所有問題；批次品質檢查通常應回傳結構化結果。

**延伸練習**：
- 將每條規則封裝成函式，統一回傳規則名稱、異常筆數與異常樣本。
- 為不同資料表設定允許的缺失比例門檻。
- 將品質摘要輸出為 CSV，並加入每日監控流程。

## 重點整理

- 主鍵唯一性與外鍵完整性是關聯式資料最基本的品質要求。
- 數值範圍、類別集合與跨欄位條件都可轉換成明確規則。
- 缺失值是否合理，必須依欄位定義與業務情境判斷。
- 資料品質不只包含格式，也包含業務邏輯、樣本代表性與時間順序。
- 品質檢查最好同時提供摘要與可追查的異常資料。